# Devoir 3

In [40]:
import qutip as qt
import numpy as np
import matplotlib.pyplot as plt

TRIANGLE = { 
    "couleurs": [("R", "V"), ("V", "B"), ("B", "R")], 
    "aretes": [(0, 1), (1, 2), (2, 0)] 
}

MAISON = { 
    "couleurs": [("R", "V"), ("V", "B"), ("R", "B"), 
                 ("V", "B"), ("R", "V")], 
    "aretes": [(0, 1), (1, 2), (2, 3), 
               (3, 0), (2, 4), (3, 4)] 
}

## 4.1 Calculs analytiques [8 points] (Max)

### a) Phase et interférence [2 points] 

In [41]:
# Pour l'état |0>, il s'agit du prod. tens. |0> x |0> x |0> où |0> = (1 0) 
base0 = qt.basis(2, 0)

# Pour 1 qubit

état0 = base0

# Porte Hadamard
H = qt.Qobj([[1, 1], [1, -1]]) / np.sqrt(2)

état0_après_H = H*état0

phi = np.pi/3 # angle arbitraire

comparaison = np.sin(phi/2)**2

# Porte P
P = qt.Qobj([[1, 0], [0, np.exp(1j * phi)]])

état0_après_P = P * état0_après_H

état0_après_H2 = H * état0_après_P

print(f"État final après 1e H : {np.round(état0_après_H.full().ravel(), 3)}")
print(f"Probabilité finale après 1e H : {np.abs(état0_après_H.full().ravel())**2}")

print(f"État final après P : {np.round(état0_après_P.full().ravel(), 3)}")
print(f"Probabilité finale après P : {np.abs(état0_après_P.full().ravel())**2}") 

print(f"État final après 2e H : {np.round(état0_après_H2.full().ravel(), 3)}")
print(f"Probabilité finale après 2e H : {np.abs(état0_après_H2.full().ravel())**2}")

print(f"Résultat comparatif : {np.round(comparaison, 2)}")

État final après 1e H : [0.707+0.j 0.707+0.j]
Probabilité finale après 1e H : [0.5 0.5]
État final après P : [0.707+0.j    0.354+0.612j]
Probabilité finale après P : [0.5 0.5]
État final après 2e H : [0.75+0.433j 0.25-0.433j]
Probabilité finale après 2e H : [0.75 0.25]
Résultat comparatif : 0.25


C'est avec la deuxième porte Hadamard que l'on modifie les probabilités d'observé |0> ou |1>. L'application d'une différence de phase relative P(phi) ne change pas les probabilités des observables.

### b) Action de la diffusion  [2 points]

In [42]:
# Pour 2 états de 1 qubit

N = 2**1 # 2**n

a0_ini = 5/2 # valeur arbitraire
a1_ini = 1 # valeur arbitraire

mu = (a0_ini + a1_ini)/N

a0_fin = 2*mu - a0_ini
a1_fin = 2*mu - a1_ini

base0 = qt.basis(2, 0)
base1 = qt.basis(2, 1)

état0 = base0
état1 = base1

psi_avant_D = a0_ini * état0 + a1_ini * état1

# Opérateur de diffusion

D = H * (2 * état0 * état0.dag() - qt.qeye(2)) * H

psi_après_D = D * psi_avant_D

print(f"État final avant D : {np.round(psi_avant_D.full().ravel(), 3)}")
print(f"a_x avant D : {a0_ini}, {a1_ini}")
print(f"État final après D : {np.round(psi_après_D.full().ravel(), 3)}")
print(f"a_x après D : {a0_fin}, {a1_fin}")


État final avant D : [2.5+0.j 1. +0.j]
a_x avant D : 2.5, 1
État final après D : [1. +0.j 2.5+0.j]
a_x après D : 1.0, 2.5


### c) Une itération à la main  [4 points]

En partant de A à C, les chaînes possibles sont (RBV):(000), (RBB):(001), (RRV):(010), (RRB):(011), (VBV):(100), (VBB):(101), (VRV):(110) et (VRB):(111). Les chaînes valides sont (RBV):(000) et (VRB):(111).

In [43]:
# Pour 3 qubits

n = 3
N = 2**n

def Q(k, b):
    Chaîne = [qt.qeye(2)] * n # Chaîne d'opérateurs
    Chaîne[k] = qt.basis(2, b) * qt.basis(2, b).dag() # Projecteur au bit k
    return qt.tensor(Chaîne) # Produit tensoriel de la chaîne

# Projecteur de conflit
# Pour le triangle, les projecteurs de conflit sont Cab = Q0(1)Q1(0), Cac = Q0(0)Q2(1)
# et Cbc = Q1(1)Q2(0) où les sommets partagent une couleur.

Cab = Q(0, 1) * Q(1, 0) 
Cac = Q(0, 0) * Q(2, 1)
Cbc = Q(1, 1) * Q(2, 0)

# Produit PI
I_N = qt.tensor([qt.qeye(2)] * n)
PI = (I_N - Cab) * (I_N - Cac) * (I_N - Cbc)

# L'oracle
Of = I_N - 2 * PI

# Superposition uniforme

H_n = qt.tensor([H] * n)
état0_n = qt.tensor([base0] * n)
s_avant_Of = H_n * état0_n

s_après_Of = Of * s_avant_Of

résultat_s_avant_Of = s_avant_Of.full().ravel()
résultat_s_après_Of = s_après_Of.full().ravel()

for x in range(N):
    print(format(x, f"0{n}b"), np.round(résultat_s_avant_Of[x], 3), 
          np.round(résultat_s_après_Of[x], 3))


000 (0.354+0j) (-0.354+0j)
001 (0.354+0j) (0.354+0j)
010 (0.354+0j) (0.354+0j)
011 (0.354+0j) (0.354+0j)
100 (0.354+0j) (0.354+0j)
101 (0.354+0j) (0.354+0j)
110 (0.354+0j) (0.354+0j)
111 (0.354+0j) (-0.354+0j)


Les chaînes valides sont belle et bien (RVB):(000) et (VRB):(111).

In [44]:
# Application de la diffusion
D = 2 * s_avant_Of * s_avant_Of.dag() - I_N

s_après_D = D * s_après_Of

résultat_s_après_D = s_après_D.full().ravel()

for x in range(N):
    print(format(x, f"0{n}b"), np.round(résultat_s_après_D[x], 3))

000 (0.707+0j)
001 (-0+0j)
010 (-0+0j)
011 (-0+0j)
100 (-0+0j)
101 (-0+0j)
110 (-0+0j)
111 (0.707+0j)


Après l'oracle, les amplitudes sont multipliées d'un facteur +1 ou -1 donc les amplitudes carrés font disparaître ce changement.

## 4.2 Construction des opérateurs avec QuTiP [15 points]

### a) Projecteurs locaux [2 points] (Max)

In [45]:
# Fonction projecteur_local

def projecteur_local(n, k, b):
    Chaîne = [qt.qeye(2)] * n # Chaîne d'opérateurs
    Chaîne[k] = qt.basis(2, b) * qt.basis(2, b).dag() # Projecteur au bit k
    return qt.tensor(Chaîne) # Produit tensoriel de la chaîne

n = 3

Q00 = projecteur_local(3, 0, 0)

état0 = qt.tensor([base0] * 3) # 000
état_conservé = Q00 * état0

état1 = qt.tensor(base1, base0, base0) # 100
état_annulé = Q00 * état1

print(f"État conservé (000): {état_conservé.full().ravel()}")
print(f"État annulé (100): {état_annulé.full().ravel()}")

État conservé (000): [1.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]
État annulé (100): [0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j]


### b) Contraintes et oracle [7 points] (Adam)

In [46]:
#1
def conflit(n, u, v, couleurs):

    Cuv = qt.qzero([2]*n)
    for i in range(2):
        for j in range(2):
            if couleurs[u][i] == couleurs[v][j]:
                Cuv += projecteur_local(n, u, i)*projecteur_local(n, v, j)
    return Cuv

#2
print(conflit(5, 0, 1, MAISON["couleurs"]))

#3
def oracle(graphe):
    n = len(graphe["couleurs"])
    I_N = qt.qeye([2]*n)
    PI = I_N
    for aretes in graphe["aretes"]:
        (u, v) = aretes
        PI *= I_N - conflit(n, u, v, graphe["couleurs"])
    
    Of = I_N - 2*PI
    return Of


Quantum object: dims=[[2, 2, 2, 2, 2], [2, 2, 2, 2, 2]], shape=(32, 32), type='oper', dtype=CSR, isherm=True
Qobj data =
[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]


#b) 4.

Puisque le complément du projecteur de conflit est aussi un projecteur, alors PI_val est aussi un projecteur. Regardons le conjugué hermitien de Of. OF_dague = (I_N - 2PI_val)_dague = I_N_dague - 2PI_val_dague = I_N - 2PI_val = Of. Of est hermitien. Regardons Of_dague*Of = (I_N - 2PI_val)(I_N - 2PI_val) = I_N - 2PI_val - 2PI_val + 4PI_val^2 = I_N -4PI_val + 4PI_val = I_N. Donc par définition Of est unitaire. 

In [47]:
#4
Of_triangle = oracle(TRIANGLE)
Of_triangle_dague = Of_triangle.dag()
if Of_triangle_dague*Of_triangle == qt.qeye([2]*len(TRIANGLE["couleurs"])):
    print("L'oracle est unitaire")

# Les etats des qubits |0> et |1>
b0 = qt.basis(2, 0)
b1 = qt.basis(2, 1)

# Les etats de base pour le triangle (8 etats de |000> a |111>)

base1 = qt.tensor([b0] * 3) # |000>
base2 = qt.tensor([b0, b0, b1]) # |001>
base3 = qt.tensor([b0, b1, b0]) # |010>
base4 = qt.tensor([b0, b1, b1]) # |011>
base5 = qt.tensor([b1, b0, b0]) # |100>
base6 = qt.tensor([b1, b0, b1]) # |101>
base7 = qt.tensor([b1, b1, b0]) # |110>
base8 = qt.tensor([b1, b1, b1]) # |111>

base = [base1, base2, base3, base4, base5, base6, base7, base8]
# test Of_triangle sur les etats de base

for i in range(8):
    etat = Of_triangle * base[i]
    coeff = base[i].dag() * etat
    if np.real(coeff) < 0:
        print(f"Le signe recu pour l'état |{format(i, '03b')}> est : -")
    if np.real(coeff) > 0:
        print(f"Le signe recu pour l'état |{format(i, '03b')}> est : +")






L'oracle est unitaire
Le signe recu pour l'état |000> est : -
Le signe recu pour l'état |001> est : +
Le signe recu pour l'état |010> est : +
Le signe recu pour l'état |011> est : +
Le signe recu pour l'état |100> est : +
Le signe recu pour l'état |101> est : +
Le signe recu pour l'état |110> est : +
Le signe recu pour l'état |111> est : -


Vérification indépendante à partir du dessin du triangle (a)

| etats | couleur | solution? |
| --- | --- | --- |
| 000 | (R,V,B) |   oui  |
| 001 | (R,V,R) |   non   |
| 010 | (R,B,B) |   non   |
|011 | (R,B,R) |   non   |
| 100 | (V,V,B) |   non   |
| 101 | (V,V,R) |   non   |
| 110 | (V,B,B) |   non   |
| 111 | (V,B,R) |   oui   |


### c) Diffusion et circuit complet [6 points] (Adam)

In [52]:
#1

def diffusion(n):

    base0 = qt.basis(2, 0)
    état0_n = qt.tensor([base0] * n)
    H_n = qt.tensor([H] * n)
    I_N = qt.qeye([2]*n)
    
    D = H_n * (2 * état0_n * état0_n.dag() - I_N) * H_n
    
    return D

def circuit(graphe, r):

    D = diffusion(len(graphe["couleurs"]))
    Of = oracle(graphe)
    H_n = qt.tensor([H] * len(graphe["couleurs"]))
    DOf = D * Of
    return (DOf**r)*H_n

état0_3 = qt.tensor([base0] * 3) # |000> pour le triangle

état_final_triangle = circuit(TRIANGLE, 1) * état0_3

for i in range(8):
    print(f"État final pour |{format(i, '03b')}> : {np.round(état_final_triangle.full().ravel()[i], 3)}")

État final pour |000> : (0.707+0j)
État final pour |001> : 0j
État final pour |010> : 0j
État final pour |011> : 0j
État final pour |100> : 0j
État final pour |101> : 0j
État final pour |110> : 0j
État final pour |111> : (0.707+0j)


## 4.3 Étude numérique de l’amplification [17 points]

### a) Validation classique [2 points] (Max)

Les chaînes valides 

In [ ]:
n = 5
N = 2**5

Chaînes_possibles = []

for x in range(N):
    Chaînes_possibles.append(format(x, f"0{n}b"))

print(Chaînes_possibles)

Chaînes_valides = []

for x in Chaînes_possibles:
    if x[0] == '1' and x[1] == '0':
        continue
    elif x[0] == '1' and x[3] == '0':
        continue
    elif x[1] == '1' and x[2] == '1':
        continue
    elif x[2] == '1' and x[3] == '1':
        continue
    elif x[2] == '0' and x[4] == '0':
        continue
    elif x[3] == '0' and x[4] == '1':
        continue
    else: Chaînes_valides.append(x)

print(Chaînes_valides)

['00000', '00001', '00010', '00011', '00100', '00101', '00110', '00111', '01000', '01001', '01010', '01011', '01100', '01101', '01110', '01111', '10000', '10001', '10010', '10011', '10100', '10101', '10110', '10111', '11000', '11001', '11010', '11011', '11100', '11101', '11110', '11111']
['00011', '00100', '01011', '11011']


Les chaînes valides sont (00011):(RVRBV), (00100):(RVBVR), (01011):(RBRBV) et (11011):(VBRBV).

### b) Interférences et amplification [15 points] (Djihad)